# UNIDAD 6 — Notebook 4: Despliegue del Proyecto
## Expone tu Modelo como Servicio Web

**Duracion:** 2–3 horas  
**Prerequisito:** `mi_proyecto_resultados.json` generado por U6_03

---

Este notebook es un template de FastAPI que **debes adaptar** para servir tu modelo.

Estructura que construiremos:
```
mi_proyecto_api/
    app.py          <- FastAPI app principal
    schemas.py      <- Pydantic schemas para tu dominio  
    model_loader.py <- Carga tu modelo entrenado
    Dockerfile      <- Contenedor opcional
    README.md       <- Instrucciones de uso
```

In [1]:
import json
import os
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings("ignore")


def _find_repo_root():
    p = Path.cwd()
    for _ in range(6):
        if (p / ".git").exists() or (p / "environment.yml").exists():
            return p
        p = p.parent
    return Path.cwd()


ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for _ep in [ROOT / ".env", Path(".env")]:
    if _ep.exists():
        load_dotenv(_ep, override=True)
        print(f"Variables cargadas desde {_ep}")
        break


def _load_json(path):
    p = Path(path)
    if not p.exists():
        p = ROOT / "educational_content/unit_06_integration_project" / p.name
    if p.exists():
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    return {}


mis_resultados = _load_json("mi_proyecto_resultados.json")

if mis_resultados:
    print(f"Resultados cargados: {mis_resultados.get('metrica_nombre')} = {mis_resultados.get('metrica_valor')}")
else:
    print("AVISO: no se encontro mi_proyecto_resultados.json")
    print("Ejecuta U6_03 completo (incluyendo Seccion 4) antes de continuar.")

Variables cargadas desde d:\Users\UCEMICH\Desktop\antigravity projects\IA NANOTECNOLOGIA\Antigravity-Nano-Research-Multiagentic-Core\.env
AVISO: no se encontro mi_proyecto_resultados.json
Ejecuta U6_03 completo (incluyendo Seccion 4) antes de continuar.


---
## Paso 1: Define el Contrato de tu API

Antes de escribir codigo, decide:
- Que informacion recibe tu API (inputs)
- Que devuelve (outputs)
- Cuantos endpoints necesitas (minimo: `/predict` y `/health`)

In [2]:
# ============================================================
#   [ESTUDIANTE] DEFINE EL CONTRATO DE TU API
# ============================================================

mi_contrato_api = {
    "nombre_del_servicio": "...",  # Ej: "predictor-band-gap"
    "version": "1.0.0",

    "endpoint_predict": {
        "metodo": "POST",
        "ruta": "/predict",
        "descripcion": "...",  # Ej: "Predice el band gap de un nanooxido"
        "inputs": {
            # Ej: "composicion": "string",
            # Ej: "tamano_nm": "float",
            # ... (lista los campos que recibe tu modelo)
        },
        "output": {
            # Ej: "band_gap_ev": "float",
            # Ej: "confianza": "float"
        }
    },

    "endpoint_health": {
        "metodo": "GET",
        "ruta": "/health",
        "descripcion": "Verifica que el servicio esta activo"
    }
}

print("Contrato API definido para:", mi_contrato_api["nombre_del_servicio"])
print("Inputs de prediccion:", list(mi_contrato_api["endpoint_predict"]["inputs"].keys()))

Contrato API definido para: ...
Inputs de prediccion: []


---
## Paso 2: Genera los Archivos de la API

Ejecuta la celda siguiente para generar la estructura de archivos. Luego **edita** cada archivo para adaptarlo a tu modelo.

In [3]:
# ============================================================
#   GENERADOR DE ESTRUCTURA DE API
#   Crea los archivos base que tu adaptas
# ============================================================

import textwrap, os
from pathlib import Path

api_dir = Path("mi_proyecto_api")
api_dir.mkdir(exist_ok=True)

servicio = mi_contrato_api.get("nombre_del_servicio", "mi-servicio")
inputs = mi_contrato_api["endpoint_predict"]["inputs"]
outputs = mi_contrato_api["endpoint_predict"]["output"]

# --- schemas.py ---
input_fields = "\n    ".join(
    f"{k}: {v.__name__ if isinstance(v, type) else v}" for k, v in inputs.items()
) or "    # [ESTUDIANTE: define los campos de entrada]"

output_fields = "\n    ".join(
    f"{k}: {v.__name__ if isinstance(v, type) else v}" for k, v in outputs.items()
) or "    # [ESTUDIANTE: define los campos de salida]"

schemas_code = f'''"""Schemas Pydantic para {servicio}."""
from pydantic import BaseModel


class InputData(BaseModel):
    # [ESTUDIANTE: adapta los tipos a los datos reales de tu modelo]
    {input_fields}
    pass  # Elimina este pass cuando agregues campos


class PredictionResult(BaseModel):
    {output_fields}
    modelo_version: str
    pass  # Elimina este pass cuando agregues campos
'''

# --- model_loader.py ---
model_loader_code = '''"""Carga el modelo entrenado desde disco."""
import pickle
from pathlib import Path

_model = None


def load_model():
    """Carga el modelo una sola vez (singleton)."""
    global _model
    if _model is None:
        # [ESTUDIANTE: ajusta la ruta a tu modelo guardado]
        model_path = Path("model.pkl")
        if model_path.exists():
            with open(model_path, "rb") as f:
                _model = pickle.load(f)
        else:
            # Modelo simulado para desarrollo — reemplaza con el tuyo
            class _DummyModel:
                def predict(self, features):
                    return [0.0]
            _model = _DummyModel()
    return _model
'''

# --- app.py ---
app_code = f'''"""API FastAPI para {servicio}."""
from fastapi import FastAPI, HTTPException
from schemas import InputData, PredictionResult
from model_loader import load_model

app = FastAPI(
    lifespan=lifespan,
    title="{servicio}",
    description="{mi_contrato_api['endpoint_predict'].get('descripcion', 'API de prediccion')}",
    version="{mi_contrato_api['version']}",
)


from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Patron correcto para FastAPI >= 0.93 (reemplaza @app.on_event deprecated)
    load_model()  # startup: pre-carga el modelo una sola vez
    yield
    # shutdown: libera recursos si necesario


@app.get("/health")
def health():
    """Verifica que el servicio esta operativo."""
    return {{"status": "ok", "servicio": "{servicio}"}}


@app.post("/predict", response_model=PredictionResult)
def predict(data: InputData):
    """Realiza una prediccion con los datos de entrada."""
    model = load_model()
    try:
        # [ESTUDIANTE: extrae los features de `data` en el orden que espera tu modelo]
        features = []
        # Ej: features = [data.tamano_nm, data.zeta_potential]

        resultado = model.predict([features])[0]

        # [ESTUDIANTE: construye el PredictionResult con tu output real]
        return PredictionResult(
            modelo_version="{mi_contrato_api['version']}"
            # Ej: band_gap_ev=float(resultado),
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# --- Dockerfile ---
dockerfile_code = '''FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements_api = '''fastapi>=0.111.0
uvicorn[standard]>=0.29.0
pydantic>=2.0.0
scikit-learn>=1.4.0
numpy>=1.26.0
# [ESTUDIANTE: agrega las dependencias de tu modelo]
'''

readme_api = f'''# {servicio}

API de prediccion generada por el Proyecto Integrador del curso Nanotecnologia + IA.

## Ejecutar localmente

```bash
pip install -r requirements.txt
python app.py
```

Visita http://localhost:8000/docs para la documentacion interactiva.

## Ejecutar con Docker

```bash
docker build -t {servicio} .
docker run -p 8000:8000 {servicio}
```

## Endpoints

- `GET /health` — Estado del servicio
- `POST /predict` — Prediccion (ver /docs para el schema)
'''

# Escribir archivos
archivos = {
    "app.py": app_code,
    "schemas.py": schemas_code,
    "model_loader.py": model_loader_code,
    "Dockerfile": dockerfile_code,
    "requirements.txt": requirements_api,
    "README.md": readme_api,
}

for nombre, contenido in archivos.items():
    ruta = api_dir / nombre
    ruta.write_text(contenido, encoding="utf-8")
    print(f"  Creado: {ruta}")

print(f"\nEstructura de API generada en ./{api_dir}/")
print("Edita app.py, schemas.py y model_loader.py para conectar tu modelo real.")

  Creado: mi_proyecto_api\app.py
  Creado: mi_proyecto_api\schemas.py
  Creado: mi_proyecto_api\model_loader.py
  Creado: mi_proyecto_api\Dockerfile
  Creado: mi_proyecto_api\requirements.txt
  Creado: mi_proyecto_api\README.md

Estructura de API generada en ./mi_proyecto_api/
Edita app.py, schemas.py y model_loader.py para conectar tu modelo real.


---
## Paso 3: Adapta y Prueba tu API

Ahora debes:
1. Guardar tu modelo entrenado (desde U6_03) como `model.pkl` dentro de `mi_proyecto_api/`
2. Editar `schemas.py` con los tipos correctos para TU problema
3. Editar `app.py` para extraer los features correctamente
4. Ejecutar el servidor y probar en [http://localhost:8000/docs](http://localhost:8000/docs)

In [4]:
# ============================================================
#   [ESTUDIANTE] GUARDA TU MODELO PARA LA API
#   Ejecuta cuando tengas 'model' entrenado de U6_03
# ============================================================
import pickle
from pathlib import Path

# Carga el modelo de U6_03 (si no esta en memoria, importalo aqui)
# model = ...  # tu modelo de U6_03

# Descomentar para guardar:
# model_pkl_path = Path("mi_proyecto_api/model.pkl")
# with open(model_pkl_path, "wb") as f:
#     pickle.dump(model, f)
# print(f"Modelo guardado en {model_pkl_path}")

print("Descomentar el bloque de arriba cuando tu modelo este entrenado.")

Descomentar el bloque de arriba cuando tu modelo este entrenado.


In [5]:
# ============================================================
#   SMOKE TEST IN-PROCESS
#   Prueba basica sin levantar el servidor completo
# ============================================================
import sys
sys.path.insert(0, "mi_proyecto_api")

try:
    from model_loader import load_model
    m = load_model()
    # [ESTUDIANTE: construye un ejemplo de input con tus features]
    ejemplo_features = []  # Ej: [5.0, -25.0]
    resultado = m.predict([ejemplo_features]) if ejemplo_features else ["sin features de ejemplo"]
    print("Smoke test OK")
    print(f"Input de ejemplo : {ejemplo_features}")
    print(f"Output del modelo: {resultado}")
except Exception as e:
    print(f"Smoke test FALLO: {e}")
    print("Revisa mi_proyecto_api/model_loader.py")

Smoke test OK
Input de ejemplo : []
Output del modelo: ['sin features de ejemplo']


---
## Checklist de Despliegue

Marca cada item antes de continuar con U6_05:

- [ ] `schemas.py` editado con mis tipos de datos reales
- [ ] `app.py` editado: extraccion de features correcta
- [ ] `model_loader.py` carga mi modelo real
- [ ] `model.pkl` guardado en `mi_proyecto_api/`
- [ ] El servidor corre sin errores (`python mi_proyecto_api/app.py`)
- [ ] `/health` responde `{"status": "ok"}`
- [ ] `/predict` responde con una prediccion valida (via `/docs`)
- [ ] (Opcional) Dockerfile probado con `docker build`

> **Siguiente:** [U6_05_REPORTE_EVALUACION.ipynb](U6_05_REPORTE_EVALUACION.ipynb)